# Integrated Change Detection System
## Object-Level Changes + Wall Surface Defect Analysis

This notebook demonstrates the **full integrated pipeline** combining:

| Component | Method | Output |
|---|---|---|
| Frame Alignment | ORB + Homography | Aligned image pair |
| Object-level Changes | YOLOv8 + SSIM | Added / Removed / Moved objects |
| Wall Surface Defects | YOLOv8 (fine-tuned) / PatchCore / Texture diff | New / Resolved / Persisting defects |
| Texture Analysis | SSIM + Colour Histogram | Paint change %, colour shift |
| Combined Report | Natural language + visuals | Stakeholder-ready output |

**Sources:**
- [`change_detection_system.py`](../change_detection_system.py)
- [`wall_defect_detector.py`](../wall_defect_detector.py)
- [`integrated_system.py`](../integrated_system.py)

## Setup

In [ ]:
import sys
sys.path.insert(0, '..')

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd
from IPython.display import display, HTML

from integrated_system import IntegratedSystem

print('All imports OK')

## Configuration

In [ ]:
FRAME_BEFORE  = '../Image_cc.jpg'    # Before image
FRAME_AFTER   = '../Image2_cc.png'   # After image
OUTPUT_DIR    = '../outputs'

# Object detection model
OBJECT_MODEL  = 'yolov8n.pt'

# Wall defect detection mode: 'texture' | 'yolo' | 'patchcore'
DEFECT_MODE   = 'texture'

# Fine-tuned defect model path (set after training, else None)
DEFECT_MODEL  = None  # e.g. '../runs/wall_defect_v1/weights/best.pt'

import os; os.makedirs(OUTPUT_DIR, exist_ok=True)

## System Architecture

```
Input: Before Image + After Image
           │
     ┌─────▼──────┐
     │  Alignment  │  ← ORB + Homography
     └─────┬───────┘
           │
    ┌──────┴──────┐
    │             │
    ▼             ▼
┌────────┐  ┌──────────────┐
│ Object │  │ Wall Surface │
│ Change │  │   Defect     │
│  (YOLO)│  │  Detection   │
│ + SSIM │  │ + Texture    │
└───┬────┘  └──────┬───────┘
    │              │
    └──────┬───────┘
           │
    ┌──────▼───────┐
    │   Combined   │
    │    Report    │
    └──────────────┘
```

## 1. Initialise the Integrated System

In [ ]:
system = IntegratedSystem(
    object_model=OBJECT_MODEL,
    defect_mode=DEFECT_MODE,
    defect_model=DEFECT_MODEL,
    conf_threshold=0.30
)

## 2. Load and Preview Input Images

In [ ]:
frame_a = cv2.imread(FRAME_BEFORE)
frame_b = cv2.imread(FRAME_AFTER)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(cv2.cvtColor(frame_a, cv2.COLOR_BGR2RGB))
axes[0].set_title('BEFORE', fontsize=14, fontweight='bold'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(frame_b, cv2.COLOR_BGR2RGB))
axes[1].set_title('AFTER',  fontsize=14, fontweight='bold'); axes[1].axis('off')
plt.suptitle('Input Images', fontsize=16)
plt.tight_layout(); plt.show()

## 3. Run the Full Pipeline

In [ ]:
results = system.process_frames(
    frame_a_path=FRAME_BEFORE,
    frame_b_path=FRAME_AFTER,
    output_dir=OUTPUT_DIR
)

## 4. Combined Report

In [ ]:
display(HTML(f"""
<div style="background:#1e1e2e; color:#cdd6f4; padding:20px; border-radius:10px;
            font-family:monospace; white-space:pre-wrap; font-size:14px;">
<b style='font-size:16px; color:#89b4fa;'>INTEGRATED CHANGE DETECTION REPORT</b>
{'='*60}
{results['combined_summary']}
{'='*60}
</div>
"""))

## 5. Object-Level Changes

In [ ]:
vis_a = cv2.imread(results['output_paths']['object_change_before'])
vis_b = cv2.imread(results['output_paths']['object_change_after'])

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
axes[0].imshow(cv2.cvtColor(vis_a, cv2.COLOR_BGR2RGB))
axes[0].set_title('Before – Object Detections', fontsize=13); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(vis_b, cv2.COLOR_BGR2RGB))
axes[1].set_title('After – Changes Highlighted',  fontsize=13); axes[1].axis('off')

legend = [
    mpatches.Patch(color='lime',   label='Added'),
    mpatches.Patch(color='yellow', label='Moved'),
    mpatches.Patch(color='blue',   label='Unchanged'),
    mpatches.Patch(color='red',    label='Removed (before)'),
]
plt.legend(handles=legend, loc='upper center', bbox_to_anchor=(0, -0.04),
           ncol=4, fontsize=11)
plt.suptitle('Object-Level Change Detection (YOLOv8)', fontsize=16)
plt.tight_layout(); plt.show()

## 6. Wall Surface Defect Comparison

In [ ]:
defect_vis = cv2.imread(results['output_paths']['wall_defect_comparison'])

plt.figure(figsize=(20, 8))
plt.imshow(cv2.cvtColor(defect_vis, cv2.COLOR_BGR2RGB))
plt.axis('off')

legend = [
    mpatches.Patch(color='lime',   label='NEW defect'),
    mpatches.Patch(color='red',    label='RESOLVED defect'),
    mpatches.Patch(color='yellow', label='PERSISTING defect'),
]
plt.legend(handles=legend, loc='lower center', ncol=3, fontsize=12,
           bbox_to_anchor=(0.5, -0.06))
plt.title('Wall Surface Defect Analysis – Before vs After', fontsize=16)
plt.tight_layout(); plt.show()

## 7. Texture / Paint Change Analysis

In [ ]:
texture = results['surface_changes']['texture_change']

# Metrics bar chart
metrics = {
    'SSIM Score\n(1=identical)':        texture['ssim_score'],
    'Colour Similarity\n(1=same colour)': texture['color_similarity'],
    'Stable Area\n(%)':                 round((100 - texture['change_percentage']) / 100, 4),
}

colors = ['#89b4fa' if v >= 0.85 else '#f38ba8' for v in metrics.values()]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(metrics.keys(), metrics.values(), color=colors, edgecolor='white', width=0.4)
ax.set_ylim(0, 1.1)
ax.axhline(0.85, color='gray', linestyle='--', label='Threshold (0.85)')
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Texture & Paint Change Metrics', fontsize=14)
ax.legend(fontsize=11)
for bar, val in zip(bars, metrics.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', fontsize=12, fontweight='bold')
plt.tight_layout(); plt.show()

status = 'CHANGED' if texture['texture_changed'] else 'STABLE'
color  = '#f38ba8' if texture['texture_changed'] else '#a6e3a1'
display(HTML(f'<div style="text-align:center;padding:12px;background:{color};border-radius:8px;'
             f'font-size:18px;font-weight:bold;">Wall Texture/Paint: {status}</div>'))

## 8. Integrated Summary Panel

In [ ]:
summary_panel = cv2.imread(results['output_paths']['integrated_summary'])

plt.figure(figsize=(22, 12))
plt.imshow(cv2.cvtColor(summary_panel, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.title('Integrated Change Detection – Summary Panel', fontsize=18, fontweight='bold')
plt.tight_layout(); plt.show()

## 9. Full Results Table

In [ ]:
obj = results['object_changes']
sur = results['surface_changes']
tex = sur['texture_change']

rows = [
    # Object changes
    *[{'Category': 'Object Change', 'Type': '+ Added',   'Detail': c['object'],  'Extra': str(c['bbox'])}          for c in obj['added']],
    *[{'Category': 'Object Change', 'Type': '- Removed', 'Detail': c['object'],  'Extra': str(c['bbox'])}          for c in obj['removed']],
    *[{'Category': 'Object Change', 'Type': '~ Moved',   'Detail': c['object'],  'Extra': f"dir={c['direction']}"}  for c in obj['moved']],
    # Surface defects
    *[{'Category': 'Surface Defect', 'Type': '🔴 New',       'Detail': d['display_name'], 'Extra': f"conf={d['confidence']:.2f}"}  for d in sur['new_defects']],
    *[{'Category': 'Surface Defect', 'Type': '🟢 Resolved',  'Detail': d['display_name'], 'Extra': f"conf={d['confidence']:.2f}"}  for d in sur['resolved_defects']],
    *[{'Category': 'Surface Defect', 'Type': '🟡 Persisting','Detail': p['before']['display_name'], 'Extra': f"iou={p['iou']:.2f}"} for p in sur['persisting_defects']],
    # Texture
    {'Category': 'Texture Analysis', 'Type': 'SSIM',             'Detail': str(tex['ssim_score']),       'Extra': '(1.0 = identical)'},
    {'Category': 'Texture Analysis', 'Type': 'Colour Similarity', 'Detail': str(tex['color_similarity']), 'Extra': '(1.0 = same colour)'},
    {'Category': 'Texture Analysis', 'Type': 'Changed Area',      'Detail': f"{tex['change_percentage']}%", 'Extra': ''},
]

if rows:
    df = pd.DataFrame(rows)
    display(df.style.set_caption('Integrated Detection Report')
            .set_table_styles([{'selector':'caption','props':[('font-size','15px'),('font-weight','bold')]}])
            .hide(axis='index'))
else:
    print('No changes detected.')

## 10. Saved Output Files

In [ ]:
print('Output files saved:')
for key, path in results['output_paths'].items():
    print(f'  {key:35s}: {path}')